# Phase 6 — Dynamics & Synthesis
**Steps 6.1 – 6.6** | Homophily regression, pivot events, cascade modeling, India verdict.

**Deliverables:** D4 (β coefficients) · D5 (contagion curves) · India verdict table

In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats as scipy_stats
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')

matplotlib_style = {'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                    'axes.edgecolor': '#30363d', 'text.color': 'white',
                    'axes.labelcolor': 'white', 'xtick.color': 'white',
                    'ytick.color': 'white', 'figure.dpi': 150,
                    'grid.color': '#30363d', 'grid.alpha': 0.5}
plt.rcParams.update(matplotlib_style)

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC   = os.path.join(ROOT, 'data', 'processed')
EXT    = os.path.join(ROOT, 'data', 'external')
NETS   = os.path.join(ROOT, 'results', 'networks')
PLOTS  = os.path.join(ROOT, 'results', 'plots')
TABLES = os.path.join(ROOT, 'results', 'tables')

def load_graph(name):
    with open(os.path.join(NETS, f'{name}.pkl'), 'rb') as f:
        return pickle.load(f)

G_full = load_graph('full')
print(f'Full network: {G_full.number_of_nodes()} nodes, {G_full.number_of_edges()} edges')

## Step 6.1 – Build Country-Pair Attribute Table (D4)

In [ ]:
region_df  = pd.read_csv(os.path.join(EXT, 'un_regional_groups.csv'))
income_df  = pd.read_csv(os.path.join(EXT, 'wb_income_groups.csv'))
region_map = dict(zip(region_df['country'], region_df['region']))
income_map = dict(zip(income_df['country'], income_df['income_group']))

# Colonial pairs — source: CEPII TRADHIST / COW colonial history
colonial_pairs = {
    ('United Kingdom', 'India'), ('United Kingdom', 'Pakistan'),
    ('United Kingdom', 'Bangladesh'), ('United Kingdom', 'Nigeria'),
    ('United Kingdom', 'Kenya'), ('United Kingdom', 'Ghana'),
    ('United Kingdom', 'Tanzania'), ('United Kingdom', 'Zimbabwe'),
    ('United Kingdom', 'South Africa'), ('United Kingdom', 'Egypt'),
    ('United Kingdom', 'Malaysia'), ('United Kingdom', 'Myanmar'),
    ('France', 'Senegal'), ('France', 'Mali'), ('France', 'Cameroon'),
    ('France', 'Ivory Coast'), ('France', 'Madagascar'),
    ('France', 'Algeria'), ('France', 'Morocco'), ('France', 'Tunisia'),
    ('France', 'Viet Nam'), ('France', 'Cambodia'),
    ('United States of America', 'Philippines'),
    ('Netherlands', 'Indonesia'), ('Belgium', 'Democratic Republic of the Congo'),
    ('Portugal', 'Angola'), ('Portugal', 'Mozambique'),
    ('Portugal', 'Guinea-Bissau'), ('Portugal', 'Sao Tome and Principe'),
    ('Spain', 'Cuba'), ('Spain', 'Dominican Republic'),
    ('Italy', 'Libya'), ('Italy', 'Somalia'), ('Italy', 'Ethiopia'),
}
colonial_set = set()
for a, b in colonial_pairs:
    colonial_set.add((a, b)); colonial_set.add((b, a))

# Approximate GDPs (PPP 2020, billions USD)
gdp_approx = {
    'United States of America': 20936, 'China': 14723, 'Japan': 5065,
    'Germany': 3806, 'India': 2709, 'United Kingdom': 2708, 'France': 2716,
    'Italy': 1886, 'Canada': 1644, 'South Korea': 1631, 'Russia': 1478,
    'Brazil': 1445, 'Australia': 1330, 'Spain': 1281, 'Mexico': 1090,
    'Indonesia': 1058, 'Netherlands': 910, 'Saudi Arabia': 703, 'Turkey': 720,
    'Switzerland': 703, 'Argentina': 383, 'Poland': 594, 'Belgium': 524,
    'Sweden': 537, 'Norway': 363, 'Israel': 402, 'South Africa': 335,
    'Egypt': 363, 'Pakistan': 263, 'Bangladesh': 302, 'Nigeria': 432,
    'Kenya': 98, 'Ethiopia': 96, 'Ghana': 68, 'Tanzania': 62,
}

# Build edge attribute table
edge_rows = []
for u, v, data in G_full.edges(data=True):
    same_region = int(region_map.get(u) == region_map.get(v) and region_map.get(u) is not None)
    same_income = int(income_map.get(u) == income_map.get(v) and income_map.get(u) is not None)
    colonial = int((u, v) in colonial_set)
    gdp_u = gdp_approx.get(u, np.nan)
    gdp_v = gdp_approx.get(v, np.nan)
    if not np.isnan(gdp_u) and not np.isnan(gdp_v) and gdp_u > 0 and gdp_v > 0:
        log_gdp_ratio = abs(np.log(gdp_u / gdp_v))
    else:
        log_gdp_ratio = np.nan
    edge_rows.append({
        'country_a': u, 'country_b': v,
        'agreement': data.get('weight', np.nan),
        'same_region': same_region,
        'same_income': same_income,
        'log_gdp_ratio': log_gdp_ratio,
        'colonial': colonial
    })

edge_df = pd.DataFrame(edge_rows)
print(f'Edge attribute table: {edge_df.shape}')
edge_df.head()

## Step 6.2 – OLS Regression (D4: Homophily)

In [ ]:
def run_ols(df_edge, label=''):
    """OLS: agreement ~ same_region + same_income + log_gdp_ratio."""
    sub = df_edge.dropna(subset=['agreement','same_region','same_income','log_gdp_ratio','colonial'])
    if len(sub) < 30:
        print(f'{label}: Too few rows ({len(sub)}), skipping OLS')
        return None

    y = sub['agreement']
    X = sub[['same_region','same_income','log_gdp_ratio','colonial']]

    # Standardize
    X_std = (X - X.mean()) / (X.std() + 1e-9)
    X_std = sm.add_constant(X_std)

    model = sm.OLS(y, X_std).fit(cov_type='cluster', cov_kwds={'groups': np.array([hash(a) ^ hash(b) for a, b in zip(sub['country_a'], sub['country_b'])])})
    print(f'\n{label} OLS Results:')
    print(model.summary2().tables[1])
    return model

ols_full = run_ols(edge_df, 'Full Network')

# Build temporal edge tables
ERAS = ['cold_war', 'post_cw', 'post_9_11', 'recent']
temporal_ols = {}
beta_over_time = []

for era in ERAS:
    G_era = load_graph(era)
    era_rows = []
    for u, v, data in G_era.edges(data=True):
        same_r = int(region_map.get(u) == region_map.get(v) and region_map.get(u) is not None)
        same_i = int(income_map.get(u) == income_map.get(v) and income_map.get(u) is not None)
        gdp_u  = gdp_approx.get(u, np.nan)
        gdp_v  = gdp_approx.get(v, np.nan)
        lgr = abs(np.log(gdp_u / gdp_v)) if (not np.isnan(gdp_u) and not np.isnan(gdp_v)
                                               and gdp_u > 0 and gdp_v > 0) else np.nan
        era_rows.append({'country_a':u,'country_b':v,'agreement':data.get('weight',np.nan),
                         'same_region':same_r,'same_income':same_i,'log_gdp_ratio':lgr,'colonial':int((u, v) in colonial_set)})
    era_edge_df = pd.DataFrame(era_rows)
    model = run_ols(era_edge_df, era)
    if model is not None:
        params = model.params
        pvals  = model.pvalues
        temporal_ols[era] = model
        beta_over_time.append({
            'era': era,
            'b_same_region': params.get('same_region', np.nan),
            'b_same_income': params.get('same_income', np.nan),
            'b_log_gdp_ratio': params.get('log_gdp_ratio', np.nan),
            'p_same_region': pvals.get('same_region', np.nan),
            'p_same_income': pvals.get('same_income', np.nan),
            'R2': model.rsquared
        })

In [ ]:
if beta_over_time:
    beta_df = pd.DataFrame(beta_over_time)
    beta_df.to_csv(os.path.join(TABLES, 'p6_homophily_D4.csv'), index=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.suptitle('Homophily β Coefficients Over Time (D4)', color='white', fontsize=13)
    era_labels = ['Cold War', 'Post-CW', 'Post-9/11', 'Recent']
    x = range(len(beta_df))
    ax.plot(x, beta_df['b_same_region'], 'o-', color='#58a6ff', lw=2, ms=8, label='β(same region)')
    ax.plot(x, beta_df['b_same_income'], 's-', color='#f85149', lw=2, ms=8, label='β(same income)')
    ax.plot(x, beta_df['b_log_gdp_ratio'], '^-', color='#3fb950', lw=2, ms=8, label='β(log GDP ratio)')
    ax.axhline(0, color='white', ls='--', lw=0.8, alpha=0.5)
    ax.set_xticks(list(x))
    ax.set_xticklabels(era_labels[:len(beta_df)])
    ax.set_ylabel('Standardised β coefficient')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS, 'p6_homophily_betas_D4.png'), bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(beta_df.to_string(index=False))

## Step 6.3 – Identify Pivot Events (D5)

In [ ]:
def find_pivot_events(df, country, top_n=5, sd_threshold=1.5):
    country_df = df[df['country'] == country].copy()
    if country_df.empty: return pd.DataFrame()
    issue_cols = ['me', 'co', 'hr', 'di', 'nu', 'ec']
    pivot_rows = []
    for issue in issue_cols:
        if issue not in country_df.columns: continue
        issue_df = country_df[country_df[issue] == 1]
        if len(issue_df) < 10: continue
        hist_mean = issue_df['v'].mean()
        hist_std  = issue_df['v'].std()
        if hist_std < 1e-6: continue
        sess_means = issue_df.groupby('session')['v'].mean().reset_index()
        sess_means.columns = ['session', 'mean_vote']
        sess_means['deviation'] = abs(sess_means['mean_vote'] - hist_mean) / hist_std
        sess_means['issue'] = issue
        sess_means['country'] = country
        pivots = sess_means[sess_means['deviation'] > sd_threshold]
        pivot_rows.append(pivots)
    if not pivot_rows: return pd.DataFrame()
    result = pd.concat(pivot_rows, ignore_index=True)
    year_map = df[['session','year']].drop_duplicates()
    result = result.merge(year_map, on='session', how='left')
    return result.sort_values('deviation', ascending=False).head(top_n)

In [ ]:
# Load full votes for pivot / cascade
try:
    df_full = pd.read_csv(os.path.join(PROC, 'votes_full.csv'))
except FileNotFoundError:
    print('Could not find votes_full.csv. Cascade simulation will skip.')
    df_full = pd.DataFrame()

if not df_full.empty:
    print('Finding pivot events for India...')
    india_pivots = find_pivot_events(df_full, 'India', top_n=5)
    if not india_pivots.empty:
        india_pivots.to_csv(os.path.join(TABLES, 'p6_pivot_events_D5.csv'), index=False)
        print(india_pivots.to_string(index=False))


## Step 6.4 – Measure Cascade Radius (D5)

In [ ]:
def compute_cascade_radius(df, G, pivot_country, pivot_session, n_hops=3):
    """
    After a pivot event at session S, check if countries within k hops
    of pivot_country also changed their vote in session S+1.
    Returns cascade decay df.
    """
    sessions = sorted(df['session'].unique())
    if pivot_session not in sessions or pivot_country not in G:
        return None

    idx = sessions.index(pivot_session)
    if idx + 1 >= len(sessions):
        return None
    next_sess = sessions[idx + 1]

    # Pivot country vote in S vs S+1
    v_s   = df[(df['country'] == pivot_country) & (df['session'] == pivot_session)]['v'].mean()
    v_s1  = df[(df['country'] == pivot_country) & (df['session'] == next_sess)]['v'].mean()
    delta_pivot = v_s1 - v_s

    # For each hop level, compute fraction of countries that 'updated'
    rows = []
    for hop in range(1, n_hops+1):
        nodes_at_hop = set()
        for n in G.nodes:
            try:
                paths = nx.shortest_path_length(G, pivot_country, n)
                if paths == hop:
                    nodes_at_hop.add(n)
            except nx.NetworkXNoPath:
                pass

        updated = 0
        total   = 0
        for neighbor in nodes_at_hop:
            v_n_s  = df[(df['country'] == neighbor) & (df['session'] == pivot_session)]['v'].mean()
            v_n_s1 = df[(df['country'] == neighbor) & (df['session'] == next_sess)]['v'].mean()
            if np.isnan(v_n_s) or np.isnan(v_n_s1):
                continue
            total += 1
            # 'updated' = moved in the same direction as pivot
            if delta_pivot != 0 and (v_n_s1 - v_n_s) * delta_pivot > 0:
                updated += 1

        rows.append({'hop': hop, 'n_countries': total,
                     'updated': updated,
                     'frac_updated': updated/total if total > 0 else np.nan})
    return pd.DataFrame(rows)


# Run for India's top pivot event (if found)
india_pivots_df = pd.read_csv(os.path.join(TABLES, 'p6_pivot_events_D5.csv')) if os.path.exists(
    os.path.join(TABLES, 'p6_pivot_events_D5.csv')) else pd.DataFrame()

cascade_results = []
if not india_pivots_df.empty:
    for _, row in india_pivots_df.iterrows():
        country = row['country']
        session = int(row['session'])
        print(f'\nCascade analysis: {country}, session {session} (year ~{row.get("year","?")})')
        # Use a small subgraph for speed
        gcc = max(nx.connected_components(G_full), key=len)
        G_sub = G_full.subgraph(list(gcc)).copy()
        cascade_df = compute_cascade_radius(df_full, G_sub, country, session, n_hops=3)
        if cascade_df is not None and not cascade_df.empty:
            cascade_df['country'] = country
            cascade_df['session'] = session
            cascade_results.append(cascade_df)
            print(cascade_df.to_string(index=False))
        if len(cascade_results) >= 3:
            break

if cascade_results:
    cascade_all_df = pd.concat(cascade_results, ignore_index=True)
    cascade_all_df.to_csv(os.path.join(TABLES, 'p6_cascade_D5.csv'), index=False)

    # Plot contagion decay curve
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.suptitle('Vote Contagion Decay Curve (D5)', color='white', fontsize=13)

    for (country, sess), grp in cascade_all_df.groupby(['country','session']):
        label = f'{country}, sess {sess}'
        ax.plot(grp['hop'], grp['frac_updated'], 'o-', ms=7, lw=2, label=label)

    ax.set_xlabel('Network distance from pivot country (hops)')
    ax.set_ylabel('Fraction of countries updating vote')
    ax.set_xticks([1, 2, 3])
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS, 'p6_cascade_D5.png'), bbox_inches='tight', facecolor='#0d1117')
    plt.show()

## Step 6.5 – India Network Profile (Synthesis)

In [ ]:
# Gather all India metrics
india_profile = {}

# Centrality ranks
centrality_df = pd.read_csv(os.path.join(TABLES, 'p3_centrality.csv'))
india_cent    = centrality_df[centrality_df['country'] == 'India']
if not india_cent.empty:
    india_profile['degree_rank']      = int(india_cent['degree_rank'].values[0])
    india_profile['betweenness_rank'] = int(india_cent['betweenness_rank'].values[0])
    india_profile['eigenvector_rank'] = int(india_cent['eigenvector_rank'].values[0])

# Fragility rank
frag_df  = pd.read_csv(os.path.join(TABLES, 'p5_fragility_index_D3.csv'))
india_fr = frag_df[frag_df['country'] == 'India']
if not india_fr.empty:
    india_profile['fragility_rank'] = int(india_fr['rank'].values[0])
    india_profile['delta_S'] = round(float(india_fr['delta_S'].values[0]), 5)

# Community memberships per issue (D1)
d1_df = pd.read_csv(os.path.join(TABLES, 'p4_india_alignment_D1.csv'))
india_profile['issue_communities'] = dict(zip(d1_df['issue_name'], d1_df['india_community']))

# NMI values (D2)
ami_df = pd.read_csv(os.path.join(TABLES, 'p4_ami.csv'))
india_profile['nmi_transitions'] = dict(zip(ami_df['transition'], ami_df['AMI']))

# Homophily β (D4)
beta_df_path = os.path.join(TABLES, 'p6_homophily_D4.csv')
if os.path.exists(beta_df_path):
    beta_df = pd.read_csv(beta_df_path)
    india_profile['homophily_betas'] = beta_df[['era','b_same_region','b_same_income']].to_dict('records')

# Build verdict table
verdict_rows = [
    ['Degree centrality rank',  india_profile.get('degree_rank', 'N/A'),
     'Higher rank = more connected'],
    ['Betweenness centrality rank', india_profile.get('betweenness_rank', 'N/A'),
     'Higher rank = more bridge-role'],
    ['Eigenvector centrality rank', india_profile.get('eigenvector_rank', 'N/A'),
     'Higher rank = better connected neighbours'],
    ['Fragility rank (ΔS)', india_profile.get('fragility_rank', 'N/A'),
     f"ΔS={india_profile.get('delta_S','?')}"],
]

for issue_name, comm in india_profile.get('issue_communities', {}).items():
    verdict_rows.append([f'Community in {issue_name}', f'Community {comm}', ''])

verdict_df = pd.DataFrame(verdict_rows, columns=['Metric', 'India Value', 'Notes'])
verdict_df.to_csv(os.path.join(TABLES, 'p6_india_verdict.csv'), index=False)

print('=' * 70)
print('INDIA NETWORK PROFILE — VERDICT TABLE')
print('=' * 70)
print(verdict_df.to_string(index=False))

In [ ]:
# India betweenness rank across eras
era_ranks = []
for era in ['full', 'cold_war', 'post_cw', 'post_9_11', 'recent']:
    G_era = load_graph(era)
    if 'India' not in G_era:
        era_ranks.append({'era': era, 'india_btw_rank': np.nan, 'india_deg_rank': np.nan})
        continue
    btw = nx.betweenness_centrality(G_era, weight='weight', normalized=True)
    deg = nx.degree_centrality(G_era)
    sorted_btw = sorted(btw, key=btw.get, reverse=True)
    sorted_deg = sorted(deg, key=deg.get, reverse=True)
    era_ranks.append({
        'era': era,
        'india_btw_rank': sorted_btw.index('India') + 1,
        'india_deg_rank': sorted_deg.index('India') + 1
    })
    print(f'{era:15s}: btw_rank={era_ranks[-1]["india_btw_rank"]:>4}, deg_rank={era_ranks[-1]["india_deg_rank"]:>4}')

era_rank_df = pd.DataFrame(era_ranks)
era_rank_df.to_csv(os.path.join(TABLES, 'p6_india_era_ranks.csv'), index=False)

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle("India's Centrality Rank Across Eras", color='white', fontsize=13)
era_names = ['Full','CW','Post-CW','Post-9/11','Recent']
x = range(len(era_rank_df))
ax.plot(x, era_rank_df['india_btw_rank'], 'o-', color='#58a6ff', lw=2, ms=8, label='Betweenness rank')
ax.plot(x, era_rank_df['india_deg_rank'], 's-', color='#f85149', lw=2, ms=8, label='Degree rank')
ax.invert_yaxis()  # Lower rank = better
ax.set_xticks(list(x))
ax.set_xticklabels(era_names[:len(era_rank_df)])
ax.set_ylabel('Rank (lower = more central)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p6_india_ranks.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

## Step 6.6 – Final Summary & Visualisation Compilation

In [ ]:
import glob
all_plots = sorted(glob.glob(os.path.join(PLOTS, '*.png')))
all_tables = sorted(glob.glob(os.path.join(TABLES, '*.csv')))

print('=== ALL PLOTS GENERATED ===')
for p in all_plots:
    print(' ', os.path.basename(p))

print('\n=== ALL TABLES GENERATED ===')
for t in all_tables:
    print(' ', os.path.basename(t))

In [ ]:
# Compile a master summary figure (4 key plots in one)
from PIL import Image
import matplotlib.image as mpimg

key_figs = [
    ('p2_full_network.png', 'Network Structure'),
    ('p3_degree_distribution.png', 'Degree Distribution'),
    ('p4_communities_full.png', 'Community Detection'),
    ('p5_fragility_index.png', 'Fragility Index (D3)')
]

available_figs = [(fn, lbl) for fn, lbl in key_figs
                  if os.path.exists(os.path.join(PLOTS, fn))]

if available_figs:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('UNGA Voting Network — Key Results Summary', color='white', fontsize=15)
    axes = axes.flatten()
    for i, (fn, lbl) in enumerate(available_figs[:4]):
        img = mpimg.imread(os.path.join(PLOTS, fn))
        axes[i].imshow(img)
        axes[i].set_title(lbl, color='white', fontsize=10)
        axes[i].axis('off')
    for j in range(len(available_figs), 4):
        axes[j].axis('off')
    plt.tight_layout()
    summary_fig_path = os.path.join(PLOTS, 'p6_master_summary.png')
    plt.savefig(summary_fig_path, bbox_inches='tight', facecolor='#0d1117', dpi=150)
    plt.show()
    print('Master summary figure saved:', summary_fig_path)
else:
    print('Run earlier notebooks first to generate plots')

## ✅ Phase 6 & Project Complete!

### Deliverables Summary
| ID | Deliverable | File(s) |
|----|-------------|---------|
| D1 | India Multi-Alignment Matrix | `results/tables/p4_india_alignment_D1.csv` |
| D2 | Alluvial / Sankey Diagram | `results/plots/p4_sankey_D2.html` |
| D3 | Superpower Fragility Index | `results/tables/p5_fragility_index_D3.csv`, `results/plots/p5_*.png` |
| D4 | Homophily β coefficients | `results/tables/p6_homophily_D4.csv`, `results/plots/p6_homophily_betas_D4.png` |
| D5 | Contagion / cascade curves | `results/tables/p6_cascade_D5.csv`, `results/plots/p6_cascade_D5.png` |

**India Verdict:** See `results/tables/p6_india_verdict.csv`